# PjBL 4 - Jogador de Xadrez Automático
### Inteligência Artificial para Xadrez com Minimax e Poda Alfa-Beta

Integrantes do grupo:
- Kevin Mitsuo Lohmann Abe
- Vinícius Pelissari Jordani
- Yasmin Cardoso Magagnin


Neste trabalho a gente fez um jogador de xadrez em Python que escolhe os lances com Minimax e poda Alfa-Beta. O tabuleiro, as regras (roque, en passant, promoção, empates, xeque-mate) e a busca foram escritas por nós. A biblioteca python-chess é usada apenas para desenhar o tabuleiro.

A IA avalia as posições por material, posição das peças, mobilidade, segurança do rei e estrutura de peões, e limita o tempo de cada jogada para nunca passar dos 2 minutos. No fim do notebook dá para jogar contra ela pelo teclado.

## 1. Constantes e representação do tabuleiro

Usamos um tabuleiro "mailbox" 10x12 (uma lista de 120 posições). As casas reais ficam nos índices 21 a 98 e o resto é uma "moldura" de casas inválidas (`FORA`). Isso evita ter que checar se um lance sai do tabuleiro: é só olhar se a casa é `FORA`.

- Peças brancas são números positivos e pretas são negativos (1 = peão, 2 = cavalo, 3 = bispo, 4 = torre, 5 = dama, 6 = rei).
- `a1 = 91`, `h1 = 98`, `a8 = 21`, `h8 = 28`. Andar para frente com as brancas é `-10`.
- Um lance é um número inteiro com os campos empacotados em bits (origem, destino, peça de promoção e um "flag" para en passant, roque e avanço duplo do peão).

In [38]:
import time
import random
import re
import sys
import chess
import chess.svg
from IPython.display import display, SVG

BRANCO, PRETO = 1, -1
PEAO, CAVALO, BISPO, TORRE, DAMA, REI = 1, 2, 3, 4, 5, 6
FORA = 100

# todas as casas reais do tabuleiro (linha 0 = oitava fileira)
CASAS = [21 + 10 * r + c for r in range(8) for c in range(8)]

DIR_CAVALO = (-21, -19, -12, -8, 8, 12, 19, 21)
DIR_DIAG = (-11, -9, 9, 11)
DIR_RETA = (-10, -1, 1, 10)
DIR_REI = DIR_DIAG + DIR_RETA

# flags dos lances
F_NORMAL, F_EP, F_ROQUE, F_DUPLO = 0, 1, 2, 3

# direitos de roque (bits)
R_BK, R_BQ, R_PK, R_PQ = 1, 2, 4, 8

NOMES_PECA = {1: "peão", 2: "cavalo", 3: "bispo", 4: "torre", 5: "dama", 6: "rei"}
LETRA_SAN = {1: "", 2: "N", 3: "B", 4: "R", 5: "Q", 6: "K"}
LETRA_FEN = {1: "p", 2: "n", 3: "b", 4: "r", 5: "q", 6: "k"}
LETRA_PROMO = {2: "n", 3: "b", 4: "r", 5: "q"}


def nome_casa(sq):
    return "abcdefgh"[(sq - 21) % 10] + str(8 - (sq - 21) // 10)


def casa_de_nome(s):
    return 91 - 10 * (int(s[1]) - 1) + "abcdefgh".index(s[0])


# chaves de Zobrist para identificar posições (tabela de transposição e repetição)
_rng = random.Random(20261113)
Z_PECA = [[_rng.getrandbits(64) for _ in range(120)] for _ in range(13)]
Z_VEZ = _rng.getrandbits(64)
Z_ROQUE = [_rng.getrandbits(64) for _ in range(16)]
Z_EP = [_rng.getrandbits(64) for _ in range(10)]

# quando uma torre/rei sai (ou é capturada) da casa, perde direito de roque
MASK_ROQUE = [15] * 120
MASK_ROQUE[91] = 15 & ~R_BQ
MASK_ROQUE[98] = 15 & ~R_BK
MASK_ROQUE[95] = 15 & ~(R_BK | R_BQ)
MASK_ROQUE[21] = 15 & ~R_PQ
MASK_ROQUE[28] = 15 & ~R_PK
MASK_ROQUE[25] = 15 & ~(R_PK | R_PQ)


def uci(m):
    """Converte o lance (inteiro) para texto, ex: 'e2e4' ou 'e7e8q'."""
    o, d, promo = m & 255, (m >> 8) & 255, (m >> 16) & 15
    return nome_casa(o) + nome_casa(d) + (LETRA_PROMO[promo] if promo else "")

## 2. Regras do xadrez (classe `Tabuleiro`)

A classe guarda a posição, gera os lances, faz e desfaz lances (`faz` / `desfaz`) e sabe dizer se o jogo acabou.

A geração é *pseudo-legal*: primeiro gera tudo que as peças conseguem fazer e depois descarta os lances que deixam o próprio rei em xeque. Assim a validação de lances ilegais (rei em xeque, peça cravada, roque passando por casa atacada, en passant que abre xeque) fica toda no mesmo lugar.

In [39]:
class Tabuleiro:
    FEN_INICIAL = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"

    def __init__(self, fen=None):
        self.carregar_fen(fen or self.FEN_INICIAL)

    # ---------- FEN ----------
    def carregar_fen(self, fen):
        partes = fen.split()
        if len(partes) < 2:
            raise ValueError("FEN incompleto")
        b = [FORA] * 120
        for sq in CASAS:
            b[sq] = 0
        mapa = {"p": 1, "n": 2, "b": 3, "r": 4, "q": 5, "k": 6}
        linhas = partes[0].split("/")
        if len(linhas) != 8:
            raise ValueError("FEN precisa de 8 fileiras")
        for i, linha in enumerate(linhas):
            c = 0
            for ch in linha:
                if ch.isdigit():
                    c += int(ch)
                else:
                    v = mapa[ch.lower()]
                    b[21 + 10 * i + c] = v if ch.isupper() else -v
                    c += 1
            if c != 8:
                raise ValueError("fileira do FEN com tamanho errado")
        self.b = b
        self.vez = BRANCO if partes[1] == "w" else PRETO
        self.roque = 0
        if len(partes) > 2:
            for ch in partes[2]:
                self.roque |= {"K": R_BK, "Q": R_BQ, "k": R_PK, "q": R_PQ}.get(ch, 0)
        self.ep = 0
        if len(partes) > 3 and partes[3] != "-":
            self.ep = casa_de_nome(partes[3])
        self.meio = int(partes[4]) if len(partes) > 4 else 0
        self.num = int(partes[5]) if len(partes) > 5 else 1
        self.rei = {}
        for sq in CASAS:
            if b[sq] == REI:
                if BRANCO in self.rei:
                    raise ValueError("FEN com dois reis brancos")
                self.rei[BRANCO] = sq
            elif b[sq] == -REI:
                if PRETO in self.rei:
                    raise ValueError("FEN com dois reis pretos")
                self.rei[PRETO] = sq
        if len(self.rei) != 2:
            raise ValueError("FEN precisa de um rei de cada cor")
        if self.atacada(self.rei[-self.vez], self.vez):
            raise ValueError("posição inválida: o lado que não joga está em xeque")
        self.pilha = []
        self.h = self.calcular_hash()
        self.hist = [self.h]

    def para_fen(self):
        linhas = []
        for r in range(8):
            s, vazios = "", 0
            for c in range(8):
                p = self.b[21 + 10 * r + c]
                if p == 0:
                    vazios += 1
                else:
                    if vazios:
                        s += str(vazios)
                        vazios = 0
                    ch = LETRA_FEN[abs(p)]
                    s += ch.upper() if p > 0 else ch
            if vazios:
                s += str(vazios)
            linhas.append(s)
        rq = ""
        for bit, ch in ((R_BK, "K"), (R_BQ, "Q"), (R_PK, "k"), (R_PQ, "q")):
            if self.roque & bit:
                rq += ch
        return "%s %s %s %s %d %d" % ("/".join(linhas), "w" if self.vez == BRANCO else "b",
                                      rq or "-", nome_casa(self.ep) if self.ep else "-",
                                      self.meio, self.num)

    def calcular_hash(self):
        h = 0
        for sq in CASAS:
            p = self.b[sq]
            if p:
                h ^= Z_PECA[p + 6][sq]
        if self.vez == PRETO:
            h ^= Z_VEZ
        h ^= Z_ROQUE[self.roque]
        if self.ep:
            h ^= Z_EP[self.ep % 10]
        return h

    def copiar(self):
        c = Tabuleiro.__new__(Tabuleiro)
        c.b = self.b[:]
        c.vez, c.roque, c.ep = self.vez, self.roque, self.ep
        c.meio, c.num = self.meio, self.num
        c.rei = dict(self.rei)
        c.h = self.h
        c.hist = self.hist[:]
        c.pilha = []
        return c

    # ---------- ataques ----------
    def atacada(self, sq, por):
        """A casa sq está sendo atacada por alguma peça da cor 'por'?"""
        b = self.b
        if por == BRANCO:
            if b[sq + 9] == PEAO or b[sq + 11] == PEAO:
                return True
        else:
            if b[sq - 9] == -PEAO or b[sq - 11] == -PEAO:
                return True
        cav, rei = por * CAVALO, por * REI
        for d in DIR_CAVALO:
            if b[sq + d] == cav:
                return True
        for d in DIR_REI:
            if b[sq + d] == rei:
                return True
        bis, tor, dam = por * BISPO, por * TORRE, por * DAMA
        for d in DIR_DIAG:
            s = sq + d
            while b[s] == 0:
                s += d
            if b[s] == bis or b[s] == dam:
                return True
        for d in DIR_RETA:
            s = sq + d
            while b[s] == 0:
                s += d
            if b[s] == tor or b[s] == dam:
                return True
        return False

    def em_xeque(self):
        return self.atacada(self.rei[self.vez], -self.vez)

    # ---------- geração de lances ----------
    def gerar(self, so_capturas=False):
        """Lances pseudo-legais (ainda podem deixar o rei em xeque)."""
        b, vez = self.b, self.vez
        mv = []
        add = mv.append
        for o in CASAS:
            p = b[o]
            if p * vez <= 0:
                continue
            tipo = p if p > 0 else -p
            if tipo == PEAO:
                if vez == BRANCO:
                    f, ini, pre = -10, 81 <= o <= 88, 31 <= o <= 38
                else:
                    f, ini, pre = 10, 31 <= o <= 38, 81 <= o <= 88
                d = o + f
                if b[d] == 0:
                    if pre:
                        for pr in ((5,) if so_capturas else (5, 4, 3, 2)):
                            add(o | d << 8 | pr << 16)
                    elif not so_capturas:
                        add(o | d << 8)
                        if ini and b[d + f] == 0:
                            add(o | (d + f) << 8 | F_DUPLO << 20)
                for dd in (d - 1, d + 1):
                    c = b[dd]
                    if c != 0 and c != FORA and c * vez < 0:
                        if pre:
                            for pr in ((5,) if so_capturas else (5, 4, 3, 2)):
                                add(o | dd << 8 | pr << 16)
                        else:
                            add(o | dd << 8)
                    elif c == 0 and dd == self.ep:
                        add(o | dd << 8 | F_EP << 20)
            elif tipo == CAVALO:
                for dd in DIR_CAVALO:
                    d = o + dd
                    c = b[d]
                    if c == 0:
                        if not so_capturas:
                            add(o | d << 8)
                    elif c != FORA and c * vez < 0:
                        add(o | d << 8)
            elif tipo == REI:
                for dd in DIR_REI:
                    d = o + dd
                    c = b[d]
                    if c == 0:
                        if not so_capturas:
                            add(o | d << 8)
                    elif c != FORA and c * vez < 0:
                        add(o | d << 8)
            else:
                dirs = DIR_DIAG if tipo == BISPO else DIR_RETA if tipo == TORRE else DIR_REI
                for dd in dirs:
                    d = o + dd
                    while True:
                        c = b[d]
                        if c == 0:
                            if not so_capturas:
                                add(o | d << 8)
                        else:
                            if c != FORA and c * vez < 0:
                                add(o | d << 8)
                            break
                        d += dd
        if not so_capturas and self.roque:
            self._gerar_roque(mv)
        return mv

    def _gerar_roque(self, mv):
        b, vez, r = self.b, self.vez, self.roque
        if vez == BRANCO:
            if r & (R_BK | R_BQ) and b[95] == REI and not self.atacada(95, PRETO):
                if r & R_BK and b[96] == 0 and b[97] == 0 and b[98] == TORRE \
                        and not self.atacada(96, PRETO):
                    mv.append(95 | 97 << 8 | F_ROQUE << 20)
                if r & R_BQ and b[94] == 0 and b[93] == 0 and b[92] == 0 and b[91] == TORRE \
                        and not self.atacada(94, PRETO):
                    mv.append(95 | 93 << 8 | F_ROQUE << 20)
        else:
            if r & (R_PK | R_PQ) and b[25] == -REI and not self.atacada(25, BRANCO):
                if r & R_PK and b[26] == 0 and b[27] == 0 and b[28] == -TORRE \
                        and not self.atacada(26, BRANCO):
                    mv.append(25 | 27 << 8 | F_ROQUE << 20)
                if r & R_PQ and b[24] == 0 and b[23] == 0 and b[22] == 0 and b[21] == -TORRE \
                        and not self.atacada(24, BRANCO):
                    mv.append(25 | 23 << 8 | F_ROQUE << 20)

    def legais(self):
        res = []
        cor = self.vez
        for m in self.gerar():
            self.faz(m)
            if not self.atacada(self.rei[cor], -cor):
                res.append(m)
            self.desfaz()
        return res

    def tem_legal(self):
        cor = self.vez
        for m in self.gerar():
            self.faz(m)
            ok = not self.atacada(self.rei[cor], -cor)
            self.desfaz()
            if ok:
                return True
        return False

    # ---------- fazer / desfazer ----------
    def faz(self, m):
        o, d, promo, fl = m & 255, (m >> 8) & 255, (m >> 16) & 15, m >> 20
        b = self.b
        p, cap, h = b[o], b[d], self.h
        cor = BRANCO if p > 0 else PRETO
        if self.ep:
            h ^= Z_EP[self.ep % 10]
        h ^= Z_ROQUE[self.roque]
        if fl == F_EP:
            cs = d + 10 * cor
            cap = b[cs]
            b[cs] = 0
            h ^= Z_PECA[cap + 6][cs]
        elif cap:
            h ^= Z_PECA[cap + 6][d]
        self.pilha.append((m, cap, self.roque, self.ep, self.meio, self.h))
        b[o] = 0
        h ^= Z_PECA[p + 6][o]
        n = promo * cor if promo else p
        b[d] = n
        h ^= Z_PECA[n + 6][d]
        if fl == F_ROQUE:
            ro, rd = (o + 3, o + 1) if d > o else (o - 4, o - 1)
            r = b[ro]
            b[ro] = 0
            b[rd] = r
            h ^= Z_PECA[r + 6][ro] ^ Z_PECA[r + 6][rd]
        if p == REI * cor:
            self.rei[cor] = d
        self.ep = (o + d) // 2 if fl == F_DUPLO else 0
        if self.ep:
            h ^= Z_EP[self.ep % 10]
        novo = self.roque & MASK_ROQUE[o] & MASK_ROQUE[d]
        self.roque = novo
        h ^= Z_ROQUE[novo]
        self.meio = 0 if (p == PEAO * cor or cap) else self.meio + 1
        self.vez = -self.vez
        h ^= Z_VEZ
        if self.vez == BRANCO:
            self.num += 1
        self.h = h
        self.hist.append(h)

    def desfaz(self):
        m, cap, roque, ep, meio, h = self.pilha.pop()
        self.hist.pop()
        self.vez = -self.vez
        if m == 0:  # lance nulo
            self.ep, self.meio, self.h = ep, meio, h
            return
        if self.vez == PRETO:
            self.num -= 1
        o, d, promo, fl = m & 255, (m >> 8) & 255, (m >> 16) & 15, m >> 20
        b, cor = self.b, self.vez
        p = PEAO * cor if promo else b[d]
        b[o] = p
        if fl == F_EP:
            b[d] = 0
            b[d + 10 * cor] = cap
        else:
            b[d] = cap
        if fl == F_ROQUE:
            ro, rd = (o + 3, o + 1) if d > o else (o - 4, o - 1)
            b[ro] = b[rd]
            b[rd] = 0
        if p == REI * cor:
            self.rei[cor] = o
        self.roque, self.ep, self.meio, self.h = roque, ep, meio, h

    def faz_nulo(self):
        """Passa a vez (usado só na busca, no 'null move pruning')."""
        self.pilha.append((0, 0, self.roque, self.ep, self.meio, self.h))
        h = self.h
        if self.ep:
            h ^= Z_EP[self.ep % 10]
            self.ep = 0
        self.vez = -self.vez
        self.h = h ^ Z_VEZ
        self.hist.append(self.h)

    # ---------- fim de jogo ----------
    def repetida(self):
        """A posição atual já apareceu antes (dentro dos lances reversíveis)?"""
        hist, h = self.hist, self.h
        n = len(hist)
        lim = max(0, n - 1 - self.meio)
        for i in range(n - 3, lim - 1, -2):
            if hist[i] == h:
                return True
        return False

    def material_insuficiente(self):
        bispos = {0: 0, 1: 0}
        cavalos = 0
        for sq in CASAS:
            p = self.b[sq]
            if p == 0:
                continue
            t = abs(p)
            if t in (PEAO, TORRE, DAMA):
                return False
            if t == CAVALO:
                cavalos += 1
            elif t == BISPO:
                bispos[((sq - 21) // 10 + (sq - 21) % 10) % 2] += 1
        if cavalos + bispos[0] + bispos[1] <= 1:
            return True
        if cavalos == 0 and (bispos[0] == 0 or bispos[1] == 0):
            return True  # só bispos, todos na mesma cor de casa
        return False

    def resultado(self):
        """None se o jogo continua; senão (codigo, vencedor, texto). vencedor: 1, -1 ou 0."""
        if not self.tem_legal():
            if self.em_xeque():
                venc = -self.vez
                return ("xeque-mate", venc, "Xeque-mate! Vencem as %s." % ("brancas" if venc == BRANCO else "pretas"))
            return ("afogamento", 0, "Empate por afogamento (rei sem lances e fora de xeque).")
        if self.material_insuficiente():
            return ("material", 0, "Empate por material insuficiente.")
        if self.meio >= 100:
            return ("50 lances", 0, "Empate pela regra dos 50 lances.")
        if self.hist.count(self.h) >= 3:
            return ("repeticao", 0, "Empate por tripla repetição de posição.")
        return None

    # ---------- notação ----------
    def para_san(self, m):
        """Notação algébrica padrão (ex: Nf3, exd5, O-O, e8=Q+)."""
        o, d, promo, fl = m & 255, (m >> 8) & 255, (m >> 16) & 15, m >> 20
        b = self.b
        tipo = abs(b[o])
        captura = b[d] != 0 or fl == F_EP
        if fl == F_ROQUE:
            s = "O-O" if d > o else "O-O-O"
        elif tipo == PEAO:
            s = (nome_casa(o)[0] + "x" if captura else "") + nome_casa(d)
            if promo:
                s += "=" + LETRA_SAN[promo]
        else:
            s = LETRA_SAN[tipo]
            outros = [x for x in self.legais()
                      if x != m and ((x >> 8) & 255) == d and abs(b[x & 255]) == tipo]
            if outros:
                mesma_col = any((x & 255) % 10 == o % 10 for x in outros)
                mesma_lin = any((x & 255) // 10 == o // 10 for x in outros)
                if not mesma_col:
                    s += nome_casa(o)[0]
                elif not mesma_lin:
                    s += nome_casa(o)[1]
                else:
                    s += nome_casa(o)
            if captura:
                s += "x"
            s += nome_casa(d)
        self.faz(m)
        if self.em_xeque():
            s += "#" if not self.tem_legal() else "+"
        self.desfaz()
        return s

## 3. Função heurística (avaliação do tabuleiro)

A avaliação devolve um número em centipeões **do ponto de vista das brancas** (positivo = brancas melhores). O Minimax maximiza esse valor quando é a vez das brancas e minimiza quando é a vez das pretas.

Fatores considerados:

1. **Material**: peão 100, cavalo 320, bispo 330, torre 500, dama 900.
2. **Tabelas de posição** (controle do centro e desenvolvimento): cada peça ganha ou perde pontos conforme a casa. Para o rei há uma tabela de meio-jogo (fica no canto, protegido) e outra de final (vai pro centro).
3. **Mobilidade**: cavalos, bispos e torres ganham pontos por quantidade de casas que alcançam.
4. **Segurança do rei**: escudo de peões na frente do rei (conta só no meio-jogo).
5. **Estrutura de peões**: penaliza peões dobrados e isolados, premia peões passados (principalmente no final).
6. **Extras**: par de bispos, torre em coluna aberta/semiaberta e, em finais ganhos, empurrar o rei adversário para a borda e aproximar o nosso rei (sem isso a IA não consegue dar mate de dama+rei contra rei).

A fase do jogo (meio-jogo ↔ final) é calculada pelo material restante e os valores de meio-jogo e final são misturados proporcionalmente.

In [40]:
VALOR = {1: 100, 2: 320, 3: 330, 4: 500, 5: 900, 6: 0}

# Tabelas no estilo "Simplified Evaluation Function": índice 0 = a8, índice 63 = h1,
# vistas pelo lado das brancas. Para as pretas a gente espelha verticalmente.
T_PEAO = [
    0, 0, 0, 0, 0, 0, 0, 0,
    50, 50, 50, 50, 50, 50, 50, 50,
    10, 10, 20, 30, 30, 20, 10, 10,
    5, 5, 10, 25, 25, 10, 5, 5,
    0, 0, 0, 20, 20, 0, 0, 0,
    5, -5, -10, 0, 0, -10, -5, 5,
    5, 10, 10, -20, -20, 10, 10, 5,
    0, 0, 0, 0, 0, 0, 0, 0]
T_CAVALO = [
    -50, -40, -30, -30, -30, -30, -40, -50,
    -40, -20, 0, 0, 0, 0, -20, -40,
    -30, 0, 10, 15, 15, 10, 0, -30,
    -30, 5, 15, 20, 20, 15, 5, -30,
    -30, 0, 15, 20, 20, 15, 0, -30,
    -30, 5, 10, 15, 15, 10, 5, -30,
    -40, -20, 0, 5, 5, 0, -20, -40,
    -50, -40, -30, -30, -30, -30, -40, -50]
T_BISPO = [
    -20, -10, -10, -10, -10, -10, -10, -20,
    -10, 0, 0, 0, 0, 0, 0, -10,
    -10, 0, 5, 10, 10, 5, 0, -10,
    -10, 5, 5, 10, 10, 5, 5, -10,
    -10, 0, 10, 10, 10, 10, 0, -10,
    -10, 10, 10, 10, 10, 10, 10, -10,
    -10, 5, 0, 0, 0, 0, 5, -10,
    -20, -10, -10, -10, -10, -10, -10, -20]
T_TORRE = [
    0, 0, 0, 0, 0, 0, 0, 0,
    5, 10, 10, 10, 10, 10, 10, 5,
    -5, 0, 0, 0, 0, 0, 0, -5,
    -5, 0, 0, 0, 0, 0, 0, -5,
    -5, 0, 0, 0, 0, 0, 0, -5,
    -5, 0, 0, 0, 0, 0, 0, -5,
    -5, 0, 0, 0, 0, 0, 0, -5,
    0, 0, 0, 5, 5, 0, 0, 0]
T_DAMA = [
    -20, -10, -10, -5, -5, -10, -10, -20,
    -10, 0, 0, 0, 0, 0, 0, -10,
    -10, 0, 5, 5, 5, 5, 0, -10,
    -5, 0, 5, 5, 5, 5, 0, -5,
    0, 0, 5, 5, 5, 5, 0, -5,
    -10, 5, 5, 5, 5, 5, 0, -10,
    -10, 0, 5, 0, 0, 0, 0, -10,
    -20, -10, -10, -5, -5, -10, -10, -20]
T_REI_MEIO = [
    -30, -40, -40, -50, -50, -40, -40, -30,
    -30, -40, -40, -50, -50, -40, -40, -30,
    -30, -40, -40, -50, -50, -40, -40, -30,
    -30, -40, -40, -50, -50, -40, -40, -30,
    -20, -30, -30, -40, -40, -30, -30, -20,
    -10, -20, -20, -20, -20, -20, -20, -10,
    20, 20, 0, 0, 0, 0, 20, 20,
    20, 30, 10, 0, 0, 10, 30, 20]
T_REI_FINAL = [
    -50, -40, -30, -20, -20, -30, -40, -50,
    -30, -20, -10, 0, 0, -10, -20, -30,
    -30, -10, 20, 30, 30, 20, -10, -30,
    -30, -10, 30, 40, 40, 30, -10, -30,
    -30, -10, 30, 40, 40, 30, -10, -30,
    -30, -10, 20, 30, 30, 20, -10, -30,
    -30, -30, 0, 0, 0, 0, -30, -30,
    -50, -30, -30, -30, -30, -30, -30, -50]


def _monta_tabelas():
    tabs_meio = {1: T_PEAO, 2: T_CAVALO, 3: T_BISPO, 4: T_TORRE, 5: T_DAMA, 6: T_REI_MEIO}
    tabs_fim = dict(tabs_meio)
    tabs_fim[6] = T_REI_FINAL
    mg = [[0] * 120 for _ in range(13)]
    eg = [[0] * 120 for _ in range(13)]
    for tipo in range(1, 7):
        for sq in CASAS:
            lin, col = (sq - 21) // 10, (sq - 21) % 10
            i_branco = lin * 8 + col
            i_preto = (7 - lin) * 8 + col
            mg[tipo + 6][sq] = VALOR[tipo] + tabs_meio[tipo][i_branco]
            eg[tipo + 6][sq] = VALOR[tipo] + tabs_fim[tipo][i_branco]
            mg[6 - tipo][sq] = -(VALOR[tipo] + tabs_meio[tipo][i_preto])
            eg[6 - tipo][sq] = -(VALOR[tipo] + tabs_fim[tipo][i_preto])
    return mg, eg


PST_MEIO, PST_FIM = _monta_tabelas()

BONUS_PASSADO = [0, 5, 10, 20, 35, 60, 100, 0]  # indexado pela fileira (0 = 1ª) vista do dono


def avaliar(t):
    """Avaliação estática em centipeões, do ponto de vista das brancas."""
    b = t.b
    mg = eg = 0
    fase = 0
    bispos_b = bispos_p = 0
    pesadas = 0
    mat_b = mat_p = 0
    # peões por coluna (índice = coluna + 1; as pontas 0 e 9 são sentinelas)
    cont_b = [0] * 10
    cont_p = [0] * 10
    max_lin_p = [-1] * 10   # maior fileira de peão preto na coluna
    min_lin_b = [8] * 10    # menor fileira de peão branco na coluna
    peoes_b = []
    peoes_p = []
    torres_b = []
    torres_p = []
    mob_mg = mob_eg = 0

    for sq in CASAS:
        p = b[sq]
        if p == 0:
            continue
        mg += PST_MEIO[p + 6][sq]
        eg += PST_FIM[p + 6][sq]
        t_ = p if p > 0 else -p
        if t_ == PEAO:
            col = (sq - 21) % 10 + 1
            lin = 7 - (sq - 21) // 10
            if p > 0:
                cont_b[col] += 1
                peoes_b.append((col, lin))
                if lin < min_lin_b[col]:
                    min_lin_b[col] = lin
            else:
                cont_p[col] += 1
                peoes_p.append((col, lin))
                if lin > max_lin_p[col]:
                    max_lin_p[col] = lin
            continue
        if t_ == REI:
            continue
        sinal = 1 if p > 0 else -1
        if sinal > 0:
            mat_b += VALOR[t_]
        else:
            mat_p += VALOR[t_]
        if t_ == CAVALO:
            fase += 1
            n = 0
            for d in DIR_CAVALO:
                c = b[sq + d]
                if c == 0 or (c != FORA and c * p < 0):
                    n += 1
            mob_mg += sinal * (n - 4) * 4
            mob_eg += sinal * (n - 4) * 4
        elif t_ == BISPO:
            fase += 1
            if sinal > 0:
                bispos_b += 1
            else:
                bispos_p += 1
            n = 0
            for d in DIR_DIAG:
                s = sq + d
                while b[s] == 0:
                    n += 1
                    s += d
                if b[s] != FORA and b[s] * p < 0:
                    n += 1
            mob_mg += sinal * (n - 6) * 3
            mob_eg += sinal * (n - 6) * 4
        elif t_ == TORRE:
            fase += 2
            pesadas += 1
            (torres_b if sinal > 0 else torres_p).append((sq - 21) % 10 + 1)
            n = 0
            for d in DIR_RETA:
                s = sq + d
                while b[s] == 0:
                    n += 1
                    s += d
                if b[s] != FORA and b[s] * p < 0:
                    n += 1
            mob_mg += sinal * (n - 7) * 2
            mob_eg += sinal * (n - 7) * 4
        else:  # dama
            fase += 4
            pesadas += 1

    # material insuficiente = empate
    if not peoes_b and not peoes_p and pesadas == 0 and (mat_b + mat_p) <= 330:
        return 0

    mg += mob_mg
    eg += mob_eg

    # par de bispos
    if bispos_b >= 2:
        mg += 30
        eg += 45
    if bispos_p >= 2:
        mg -= 30
        eg -= 45

    # estrutura de peões
    for col, lin in peoes_b:
        if cont_b[col] > 1:
            mg -= 10
            eg -= 15
        if cont_b[col - 1] == 0 and cont_b[col + 1] == 0:
            mg -= 12
            eg -= 15
        if max_lin_p[col - 1] <= lin and max_lin_p[col] <= lin and max_lin_p[col + 1] <= lin:
            mg += BONUS_PASSADO[lin] // 2
            eg += BONUS_PASSADO[lin]
    for col, lin in peoes_p:
        if cont_p[col] > 1:
            mg += 10
            eg += 15
        if cont_p[col - 1] == 0 and cont_p[col + 1] == 0:
            mg += 12
            eg += 15
        if min_lin_b[col - 1] >= lin and min_lin_b[col] >= lin and min_lin_b[col + 1] >= lin:
            mg -= BONUS_PASSADO[7 - lin] // 2
            eg -= BONUS_PASSADO[7 - lin]

    # torres em colunas abertas
    for col in torres_b:
        if cont_b[col] == 0:
            bonus = 20 if cont_p[col] == 0 else 10
            mg += bonus
            eg += bonus
    for col in torres_p:
        if cont_p[col] == 0:
            bonus = 20 if cont_b[col] == 0 else 10
            mg -= bonus
            eg -= bonus

    # segurança do rei: escudo de peões (só vale no meio-jogo)
    rb, rp = t.rei[BRANCO], t.rei[PRETO]
    if rb >= 81:
        esc = 0
        for dc in (-1, 0, 1):
            if b[rb - 10 + dc] == PEAO:
                esc += 10
            elif b[rb - 20 + dc] == PEAO:
                esc += 5
            else:
                esc -= 10
        mg += esc
    if rp <= 38:
        esc = 0
        for dc in (-1, 0, 1):
            if b[rp + 10 + dc] == -PEAO:
                esc += 10
            elif b[rp + 20 + dc] == -PEAO:
                esc += 5
            else:
                esc -= 10
        mg -= esc

    if fase > 24:
        fase = 24

    # final ganho: encurralar o rei adversário (necessário para dar mate)
    dif = mat_b - mat_p
    if fase <= 12 and (dif >= 300 or dif <= -300):
        if dif > 0:
            perdedor, vencedor, sinal = rp, rb, 1
        else:
            perdedor, vencedor, sinal = rb, rp, -1
        lp, cp = (perdedor - 21) // 10, (perdedor - 21) % 10
        lv, cv = (vencedor - 21) // 10, (vencedor - 21) % 10
        dist_centro = max(3 - cp, cp - 4) + max(3 - lp, lp - 4)
        dist_reis = abs(lp - lv) + abs(cp - cv)
        eg += sinal * (dist_centro * 12 + (14 - dist_reis) * 5)

    return (mg * fase + eg * (24 - fase)) // 24

## 4. Busca: Minimax com poda Alfa-Beta

A busca está escrita no formato Minimax "explícito": quando é a vez das brancas o nó **maximiza** a avaliação e quando é a vez das pretas ele **minimiza**. `alfa` é o melhor valor que o maximizador já tem garantido e `beta` o melhor que o minimizador já tem garantido. Se `alfa >= beta` o resto dos lances daquele nó não precisa ser visto (poda).

Melhorias que colocamos em cima do Minimax básico, para chegar numa profundidade maior no mesmo tempo:

- **Aprofundamento iterativo**: busca profundidade 1, depois 2, 3... e guarda o melhor lance da última profundidade completa. Se o relógio estoura no meio de uma profundidade, a gente descarta ela e usa o lance da anterior. É isso que garante o limite de tempo.
- **Tabela de transposição** (Zobrist): lembra posições já avaliadas e o melhor lance de cada uma.
- **Ordenação de lances**: primeiro o lance da tabela, depois capturas boas (MVV-LVA), promoções, lances "killer" e heurística de histórico. Quanto melhor a ordem, mais a poda corta.
- **Busca de quiescência**: no fim da profundidade continua só com capturas para não avaliar posição no meio de uma troca.
- **Extensão de xeque**: quando está em xeque, busca um nível a mais.
- **Null move pruning** e **LMR** (reduzir a profundidade dos lances quietos tardios, com re-busca se parecerem bons).
- **Livro de aberturas** pequeno, para as primeiras jogadas saírem na hora e com variedade.

In [41]:
INF = 10 ** 9
MATE = 100000
MATE_LIM = MATE - 1000
VAL_ORD = [0, 1, 3, 3, 5, 9, 20]  # valores para ordenar capturas


class TempoEsgotado(Exception):
    pass


# linhas do livro de aberturas (cada uma é uma sequência de lances)
LINHAS_LIVRO = [
    "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6 b5a4 g8f6 e1g1 f8e7",
    "e2e4 e7e5 g1f3 b8c6 f1c4 f8c5 c2c3 g8f6 d2d3 d7d6",
    "e2e4 e7e5 g1f3 b8c6 d2d4 e5d4 f3d4 g8f6 b1c3 f8b4",
    "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 f3d4 g8f6 b1c3 a7a6",
    "e2e4 c7c5 g1f3 b8c6 d2d4 c5d4 f3d4 g8f6 b1c3 e7e5",
    "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6 c1g5 f8e7 e4e5 f6d7",
    "e2e4 c7c6 d2d4 d7d5 b1c3 d5e4 c3e4 c8f5 e4g3 f5g6",
    "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6 c1g5 f8e7 e2e3 e8g8",
    "d2d4 g8f6 c2c4 e7e6 b1c3 f8b4 e2e3 e8g8 f1d3 d7d5",
    "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 e2e4 d7d6 g1f3 e8g8",
    "d2d4 d7d5 g1f3 g8f6 c2c4 e7e6 b1c3 f8e7 c1f4 e8g8",
    "g1f3 d7d5 g2g3 g8f6 f1g2 e7e6 e1g1 f8e7 d2d3 e8g8",
    "c2c4 e7e5 b1c3 g8f6 g1f3 b8c6 g2g3 d7d5 c4d5 f6d5",
    "c2c4 g8f6 b1c3 e7e6 e2e4 d7d5 e4e5 d5d4 e5f6 d4c3",
]


def _tenta_lance(t, texto):
    o, d = casa_de_nome(texto[0:2]), casa_de_nome(texto[2:4])
    prom = {"n": 2, "b": 3, "r": 4, "q": 5}.get(texto[4:5], 0)
    for m in t.legais():
        if m & 255 == o and (m >> 8) & 255 == d and (m >> 16) & 15 == prom:
            return m
    raise ValueError("lance do livro inválido: " + texto)


def _monta_livro():
    livro = {}
    for linha in LINHAS_LIVRO:
        t = Tabuleiro()
        for texto in linha.split():
            m = _tenta_lance(t, texto)
            lista = livro.setdefault(t.h, [])
            if m not in lista:
                lista.append(m)
            t.faz(m)
    return livro


LIVRO = _monta_livro()


class Motor:
    """Jogador automático: Minimax + Alfa-Beta com limite de tempo por jogada."""

    def __init__(self, tempo=10.0, usar_livro=True, nome="IA"):
        self.tempo = max(0.05, min(float(tempo), 110.0))  # nunca passa de 110 s (limite do trabalho: 120 s)
        self.usar_livro = usar_livro
        self.nome = nome
        self.tt = {}
        self.matador = [[0, 0] for _ in range(80)]
        self.historia = [0] * (13 * 120)
        self.nos = 0
        self.stats = []  # (profundidade, tempo gasto) de cada jogada

    # ---------- ordenação ----------
    def _ordenar(self, t, moves, tt_mov, ply):
        b = t.b
        k1, k2 = self.matador[ply]
        hist = self.historia
        pontuados = []
        for m in moves:
            if m == tt_mov:
                s = 10 ** 7
            else:
                o, d, promo = m & 255, (m >> 8) & 255, (m >> 16) & 15
                v = b[d]
                if v or (m >> 20) == F_EP:
                    vit = VAL_ORD[abs(v)] if v else 1
                    s = 1000000 + vit * 100 - VAL_ORD[abs(b[o])]
                elif promo:
                    s = 1100000 if promo == 5 else 500000
                elif m == k1:
                    s = 900000
                elif m == k2:
                    s = 899000
                else:
                    s = hist[(b[o] + 6) * 120 + d]
            pontuados.append((s, m))
        pontuados.sort(reverse=True)
        return [m for _, m in pontuados]

    def _tem_peca_pesada(self, t):
        b, vez = t.b, t.vez
        for sq in CASAS:
            p = b[sq] * vez
            if 2 <= p <= 5:
                return True
        return False

    # ---------- quiescência ----------
    def _quiescencia(self, alfa, beta, ply):
        t = self.t
        self.nos += 1
        if self.pode_abortar and (self.nos & 2047) == 0 and time.perf_counter() > self.limite:
            raise TempoEsgotado
        parado = avaliar(t)
        if ply >= 70:
            return parado
        b = t.b
        if t.vez == BRANCO:
            if parado >= beta:
                return parado
            if parado > alfa:
                alfa = parado
            melhor = parado
            for m in self._ordenar(t, t.gerar(True), 0, ply):
                v = b[(m >> 8) & 255]
                ganho = VALOR[abs(v)] if v else 100
                if (m >> 16) & 15:
                    ganho += 800
                if parado + ganho + 200 < alfa:  # delta pruning
                    continue
                t.faz(m)
                if t.atacada(t.rei[BRANCO], PRETO):
                    t.desfaz()
                    continue
                val = self._quiescencia(alfa, beta, ply + 1)
                t.desfaz()
                if val > melhor:
                    melhor = val
                    if val > alfa:
                        alfa = val
                        if alfa >= beta:
                            break
            return melhor
        else:
            if parado <= alfa:
                return parado
            if parado < beta:
                beta = parado
            melhor = parado
            for m in self._ordenar(t, t.gerar(True), 0, ply):
                v = b[(m >> 8) & 255]
                ganho = VALOR[abs(v)] if v else 100
                if (m >> 16) & 15:
                    ganho += 800
                if parado - ganho - 200 > beta:
                    continue
                t.faz(m)
                if t.atacada(t.rei[PRETO], BRANCO):
                    t.desfaz()
                    continue
                val = self._quiescencia(alfa, beta, ply + 1)
                t.desfaz()
                if val < melhor:
                    melhor = val
                    if val < beta:
                        beta = val
                        if alfa >= beta:
                            break
            return melhor

    # ---------- Minimax com Alfa-Beta ----------
    def _minimax(self, prof, alfa, beta, ply, nulo_ok):
        t = self.t
        self.nos += 1
        if self.pode_abortar and (self.nos & 2047) == 0 and time.perf_counter() > self.limite:
            raise TempoEsgotado
        if ply > 0 and (t.meio >= 100 or t.repetida()):
            return 0
        maximiza = t.vez == BRANCO
        em_xeque = t.em_xeque()
        if em_xeque and ply < 60:
            prof += 1
        if prof <= 0 or ply >= 70:
            return self._quiescencia(alfa, beta, ply)

        alfa0, beta0 = alfa, beta
        tt_mov = 0
        ent = self.tt.get(t.h)
        if ent:
            d_e, flag, val, tt_mov = ent
            if d_e >= prof and ply > 0:
                if val > MATE_LIM:
                    val -= ply
                elif val < -MATE_LIM:
                    val += ply
                if flag == 0:
                    return val
                if flag == 1:
                    if val > alfa:
                        alfa = val
                elif val < beta:
                    beta = val
                if alfa >= beta:
                    return val

        # null move: se até passando a vez eu já corto, nem precisa olhar os lances
        if nulo_ok and not em_xeque and prof >= 3 and ply > 0 and self._tem_peca_pesada(t):
            t.faz_nulo()
            try:
                if maximiza:
                    v = self._minimax(prof - 3, beta - 1, beta, ply + 1, False)
                else:
                    v = self._minimax(prof - 3, alfa, alfa + 1, ply + 1, False)
            finally:
                t.desfaz()
            if maximiza and v >= beta:
                return beta if abs(v) > MATE_LIM else v
            if not maximiza and v <= alfa:
                return alfa if abs(v) > MATE_LIM else v

        moves = self._ordenar(t, t.gerar(), tt_mov, ply)
        b = t.b
        legais = 0
        melhor_m = 0
        k1, k2 = self.matador[ply]
        if maximiza:
            melhor = -INF
            for m in moves:
                d = (m >> 8) & 255
                quieto = b[d] == 0 and not (m >> 16) & 15 and (m >> 20) != F_EP
                o = m & 255
                peca = b[o]
                t.faz(m)
                if t.atacada(t.rei[BRANCO], PRETO):
                    t.desfaz()
                    continue
                legais += 1
                if legais > 4 and prof >= 3 and quieto and not em_xeque and m != k1 and m != k2:
                    v = self._minimax(prof - 2, alfa, alfa + 1, ply + 1, True)
                    if v > alfa:
                        v = self._minimax(prof - 1, alfa, beta, ply + 1, True)
                else:
                    v = self._minimax(prof - 1, alfa, beta, ply + 1, True)
                t.desfaz()
                if v > melhor:
                    melhor, melhor_m = v, m
                    if ply == 0:
                        self.raiz_mov, self.raiz_val = m, v
                    if v > alfa:
                        alfa = v
                        if alfa >= beta:
                            if quieto:
                                self._registra_corte(m, peca, d, prof, ply)
                            break
        else:
            melhor = INF
            for m in moves:
                d = (m >> 8) & 255
                quieto = b[d] == 0 and not (m >> 16) & 15 and (m >> 20) != F_EP
                o = m & 255
                peca = b[o]
                t.faz(m)
                if t.atacada(t.rei[PRETO], BRANCO):
                    t.desfaz()
                    continue
                legais += 1
                if legais > 4 and prof >= 3 and quieto and not em_xeque and m != k1 and m != k2:
                    v = self._minimax(prof - 2, beta - 1, beta, ply + 1, True)
                    if v < beta:
                        v = self._minimax(prof - 1, alfa, beta, ply + 1, True)
                else:
                    v = self._minimax(prof - 1, alfa, beta, ply + 1, True)
                t.desfaz()
                if v < melhor:
                    melhor, melhor_m = v, m
                    if ply == 0:
                        self.raiz_mov, self.raiz_val = m, v
                    if v < beta:
                        beta = v
                        if alfa >= beta:
                            if quieto:
                                self._registra_corte(m, peca, d, prof, ply)
                            break

        if legais == 0:
            if em_xeque:
                return -(MATE - ply) if maximiza else (MATE - ply)
            return 0

        if melhor <= alfa0:
            flag = 2
        elif melhor >= beta0:
            flag = 1
        else:
            flag = 0
        val = melhor
        if val > MATE_LIM:
            val += ply
        elif val < -MATE_LIM:
            val -= ply
        self.tt[t.h] = (prof, flag, val, melhor_m)
        return melhor

    def _registra_corte(self, m, peca, destino, prof, ply):
        k = self.matador[ply]
        if k[0] != m:
            k[1] = k[0]
            k[0] = m
        i = (peca + 6) * 120 + destino
        self.historia[i] = min(self.historia[i] + prof * prof, 800000)

    # ---------- escolha da jogada (aprofundamento iterativo) ----------
    def escolher(self, tab):
        """Recebe um Tabuleiro e devolve (lance, info). O tabuleiro original não é alterado."""
        ini = time.perf_counter()
        info = {"profundidade": 0, "nos": 0, "valor": 0, "livro": False, "tempo": 0.0}
        legais = tab.legais()
        if not legais:
            return None, info
        if len(legais) == 1:
            info["tempo"] = time.perf_counter() - ini
            return legais[0], info
        if self.usar_livro and tab.h in LIVRO:
            opcoes = [m for m in LIVRO[tab.h] if m in legais]
            if opcoes:
                info["livro"] = True
                info["tempo"] = time.perf_counter() - ini
                return random.choice(opcoes), info

        self.t = tab.copiar()
        self.limite = ini + self.tempo
        self.nos = 0
        if len(self.tt) > 1500000:
            self.tt.clear()
        self.historia = [x // 4 for x in self.historia]
        self.matador = [[0, 0] for _ in range(80)]
        melhor = legais[0]
        valor = 0
        prof_ok = 0
        for prof in range(1, 64):
            self.pode_abortar = prof > 1  # a profundidade 1 sempre termina
            self.raiz_mov = 0
            try:
                v = self._minimax(prof, -INF, INF, 0, False)
            except TempoEsgotado:
                break
            if self.raiz_mov:
                melhor, valor, prof_ok = self.raiz_mov, v, prof
            if abs(v) > MATE_LIM:
                break
            if time.perf_counter() - ini > self.tempo * 0.4:
                break  # a próxima profundidade provavelmente não termina a tempo
        info.update(profundidade=prof_ok, nos=self.nos, valor=valor,
                    tempo=time.perf_counter() - ini)
        self.stats.append((prof_ok, info["tempo"]))
        return melhor, info


_motor_global = None


def jogada_ia(fen, tempo=10.0):
    """Interface simples para a competição: recebe um FEN, devolve o lance em texto (ex: 'e2e4')."""
    global _motor_global
    if _motor_global is None or _motor_global.tempo != min(float(tempo), 110.0):
        _motor_global = Motor(tempo)
    m, _ = _motor_global.escolher(Tabuleiro(fen))
    return uci(m) if m else None

## 5. Testes das regras (perft)

O *perft* conta quantas posições existem a N lances de distância de uma posição. Esses números são conhecidos e publicados para posições feitas justamente para quebrar implementações de xadrez (roque, en passant, promoção, cravadas, xeque descoberto). Se o nosso gerador bate com todos, as regras estão certas.

In [42]:
def perft(t, prof):
    if prof == 0:
        return 1
    n = 0
    cor = t.vez
    for m in t.gerar():
        t.faz(m)
        if not t.atacada(t.rei[cor], -cor):
            n += 1 if prof == 1 else perft(t, prof - 1)
        t.desfaz()
    return n


TESTES_PERFT = [
    ("Posição inicial", Tabuleiro.FEN_INICIAL, [20, 400, 8902, 197281]),
    ("Kiwipete (roques, cravadas)", "r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1", [48, 2039, 97862]),
    ("Final com en passant", "8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 1", [14, 191, 2812, 43238]),
    ("Promoções e xeques", "r3k2r/Pppp1ppp/1b3nbN/nP6/BBP1P3/q4N2/Pp1P2PP/R2Q1RK1 w kq - 0 1", [6, 264, 9467]),
    ("Posição 5", "rnbq1k1r/pp1Pbppp/2p5/8/2B5/8/PPP1NnPP/RNBQK2R w KQ - 1 8", [44, 1486, 62379]),
    ("Promoção das pretas", "n1n5/PPPk4/8/8/8/8/4Kppp/5N1N b - - 0 1", [24, 496, 9483]),
]


def rodar_perft():
    tudo_ok = True
    for nome, fen, esperado in TESTES_PERFT:
        t = Tabuleiro(fen)
        obtido = []
        t0 = time.perf_counter()
        for i in range(len(esperado)):
            obtido.append(perft(t, i + 1))
        ok = obtido == esperado
        tudo_ok &= ok
        print("%-30s %s  (%.1fs)" % (nome, "OK" if ok else "ERRO", time.perf_counter() - t0))
        if not ok:
            print("   esperado:", esperado, "obtido:", obtido)
        # o tabuleiro tem que voltar exatamente ao que era (hash e FEN iguais)
        assert t.para_fen() == Tabuleiro(fen).para_fen() and t.h == t.calcular_hash()
    print("\nTodas as regras conferem com os valores oficiais!" if tudo_ok else "\nTem erro nas regras, revisar!")
    return tudo_ok


rodar_perft()

Posição inicial                OK  (2.1s)
Kiwipete (roques, cravadas)    OK  (1.5s)
Final com en passant           OK  (0.3s)
Promoções e xeques             OK  (0.1s)
Posição 5                      OK  (0.4s)
Promoção das pretas            OK  (0.1s)

Todas as regras conferem com os valores oficiais!


True

### Fim de jogo, roque e outras regras

Aqui testamos casos específicos: xeque-mate, afogamento, material insuficiente, regra dos 50 lances, tripla repetição, rejeição de lances ilegais, roque proibido e en passant.

In [43]:
def teste_regras():
    def texto(t):
        r = t.resultado()
        return r[0] if r else None

    def tem(t, lance):
        return any(uci(m) == lance for m in t.legais())

    # mate do pastor / mate de corredor
    t = Tabuleiro("rnb1kbnr/pppp1ppp/8/4p3/6Pq/5P2/PPPPP2P/RNBQKBNR w KQkq - 1 3")
    assert texto(t) == "xeque-mate", "mate do louco"
    t = Tabuleiro("7k/5Q2/6K1/8/8/8/8/8 b - - 0 1")
    assert texto(t) == "afogamento", "afogamento"
    assert texto(Tabuleiro("8/8/8/4k3/8/3K4/8/8 w - - 0 1")) == "material"
    assert texto(Tabuleiro("8/8/8/4k3/8/3K4/8/6B1 w - - 0 1")) == "material"
    assert texto(Tabuleiro("8/8/8/4k3/8/3K4/8/6R1 w - - 0 1")) is None
    assert texto(Tabuleiro("4k3/8/8/8/8/8/4P3/4K3 w - - 100 80")) == "50 lances"

    # tripla repetição
    t = Tabuleiro("4k3/8/8/8/8/8/8/R3K3 w - - 0 1")
    for _ in range(2):
        for l in ("a1a2", "e8e7", "a2a1", "e7e8"):
            t.faz(_tenta_lance(t, l))
    assert texto(t) == "repeticao"

    # roque: permitido, e proibido quando passa por casa atacada, está em xeque ou perdeu o direito
    t = Tabuleiro("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1")
    assert tem(t, "e1g1") and tem(t, "e1c1")
    t = Tabuleiro("r3k2r/8/8/8/8/5r2/8/R3K2R w KQkq - 0 1")
    assert not tem(t, "e1g1") and tem(t, "e1c1")       # f1 atacada pela torre em f3
    t = Tabuleiro("r3k2r/8/8/8/8/4r3/8/R3K2R w KQkq - 0 1")
    assert not tem(t, "e1g1") and not tem(t, "e1c1")   # rei em xeque
    t = Tabuleiro("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1")
    t.faz(_tenta_lance(t, "h1g1")); t.faz(_tenta_lance(t, "a8a7"))
    t.faz(_tenta_lance(t, "g1h1")); t.faz(_tenta_lance(t, "a7a8"))
    assert not tem(t, "e1g1") and tem(t, "e1c1")       # torre já mexeu

    # en passant, inclusive o caso em que ele é ilegal (rei exposto na fileira)
    t = Tabuleiro("rnbqkbnr/ppp1pppp/8/8/3pP3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 3")
    assert tem(t, "d4e3")
    t = Tabuleiro("8/8/8/8/k2pP2R/8/8/4K3 b - e3 0 1")
    assert not tem(t, "d4e3")

    # promoção gera os 4 lances
    t = Tabuleiro("8/P6k/8/8/8/8/8/K7 w - - 0 1")
    assert sorted(uci(m) for m in t.legais() if uci(m).startswith("a7")) == ["a7a8b", "a7a8n", "a7a8q", "a7a8r"]

    # peça cravada não pode sair da linha
    t = Tabuleiro("4r1k1/8/8/8/8/8/4B3/4K3 w - - 0 1")
    assert not any(uci(m).startswith("e2") and not uci(m).startswith("e2e") for m in t.legais())
    # notação SAN
    t = Tabuleiro()
    assert t.para_san(_tenta_lance(t, "g1f3")) == "Nf3"
    t = Tabuleiro("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1")
    assert t.para_san(_tenta_lance(t, "e1g1")) == "O-O"
    print("Testes de regras: todos passaram.")


teste_regras()

Testes de regras: todos passaram.


## 6. Testes da IA (táticas e limite de tempo)

Posições onde existe um mate forçado ou uma peça de graça. A IA tem que achar o lance certo, e a gente também confere o tempo gasto e a profundidade.

In [44]:
def teste_taticas(tempo=5):
    motor = Motor(tempo, usar_livro=False)
    casos = [
        ("Mate em 1 (corredor)", "6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1", {"a1a8"}),
        ("Mate em 1 das pretas", "r5k1/8/8/8/8/8/5PPP/6K1 b - - 0 1", {"a8a1"}),
        ("Mate em 1 (mate do pastor)", "r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4", {"h5f7"}),
        ("Ganhar a dama de graça", "4k3/8/8/3q4/8/8/8/3RK3 w - - 0 1", {"d1d5"}),
        ("Não deixar a dama pendurada", "4k3/8/2p5/3q4/4P3/8/8/4K3 b - - 0 1", {"d5e4", "d5d6", "d5d8", "d5e5", "d5e6", "d5c5", "d5b5", "d5a5"}),
    ]
    for nome, fen, certos in casos:
        t = Tabuleiro(fen)
        m, info = motor.escolher(t)
        lance = uci(m)
        ok = lance in certos
        print("%-34s IA jogou %-6s %s  prof=%d  %.1fs  nós=%d" % (
            nome, lance, "OK" if ok else "(esperado %s)" % sorted(certos), info["profundidade"], info["tempo"], info["nos"]))


teste_taticas()

Mate em 1 (corredor)               IA jogou a1a8   OK  prof=1  0.0s  nós=40
Mate em 1 das pretas               IA jogou a8a1   OK  prof=1  0.0s  nós=34
Mate em 1 (mate do pastor)         IA jogou h5f7   OK  prof=1  0.0s  nós=92
Ganhar a dama de graça             IA jogou d1d5   OK  prof=13  3.1s  nós=201901
Não deixar a dama pendurada        IA jogou d5e4   OK  prof=8  2.5s  nós=164372


### Teste do limite de tempo e de finais

Em posições bem diferentes (abertura, meio-jogo cheio de peças, final) medimos o tempo real de resposta com um limite configurado. Depois a IA joga sozinha dois finais teóricos (dama+rei x rei e torre+rei x rei) para ver se consegue dar o mate de verdade.

In [45]:
def teste_tempo(limite=3.0):
    posicoes = [t[1] for t in TESTES_PERFT] + [
        "r4rk1/1pp1qppp/p1np1n2/2b1p1B1/2B1P1b1/P1NP1N2/1PP1QPPP/R4RK1 w - - 0 10",
        "2r3k1/pp3ppp/2n1b3/3p4/3P4/2N1BN2/PP3PPP/2R3K1 w - - 0 1",
    ]
    motor = Motor(limite, usar_livro=False)
    pior = 0
    for fen in posicoes:
        _, info = motor.escolher(Tabuleiro(fen))
        pior = max(pior, info["tempo"])
        print("prof %2d  %.2fs  %s" % (info["profundidade"], info["tempo"], fen[:40]))
    print("\nPior tempo: %.2fs (limite configurado %.1fs, limite do trabalho 120s)" % (pior, limite))
    assert pior < limite * 1.15 + 0.2


teste_tempo()

prof  6  1.33s  rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK
prof  4  3.00s  r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p
prof  8  1.51s  8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 
prof  4  1.65s  r3k2r/Pppp1ppp/1b3nbN/nP6/BBP1P3/q4N2/Pp
prof  6  1.26s  rnbq1k1r/pp1Pbppp/2p5/8/2B5/8/PPP1NnPP/R
prof  7  3.02s  n1n5/PPPk4/8/8/8/8/4Kppp/5N1N b - - 0 1
prof  5  1.21s  r4rk1/1pp1qppp/p1np1n2/2b1p1B1/2B1P1b1/P
prof  6  2.14s  2r3k1/pp3ppp/2n1b3/3p4/3P4/2N1BN2/PP3PPP

Pior tempo: 3.02s (limite configurado 3.0s, limite do trabalho 120s)


## 7. Partida automática IA x IA

Esta função faz o papel de árbitro: ela chama cada motor na sua vez, confere se o lance é legal, mede o tempo e aplica as regras de derrota automática do enunciado (lance ilegal ou mais de 2 minutos). Também deu pra usar para testar a estabilidade do sistema e para simular a competição entre dois motores.

In [46]:
LIMITE_TEMPO_OFICIAL = 120.0


def partida_automatica(motor_brancas, motor_pretas, fen=None, max_lances=250, mostrar=True):
    t = Tabuleiro(fen)
    motores = {BRANCO: motor_brancas, PRETO: motor_pretas}
    lances_san = []
    tempos = {BRANCO: [], PRETO: []}
    profs = {BRANCO: [], PRETO: []}
    while True:
        r = t.resultado()
        if r:
            codigo, venc, texto = r
            break
        if len(lances_san) >= 2 * max_lances:
            codigo, venc, texto = "limite", 0, "Empate: limite de lances da simulação."
            break
        cor = t.vez
        ini = time.perf_counter()
        try:
            m, info = motores[cor].escolher(t)
        except Exception as e:  # erro do motor = derrota
            codigo, venc, texto = "erro", -cor, "Derrota automática: o motor travou (%s)." % e
            break
        gasto = time.perf_counter() - ini
        if gasto > LIMITE_TEMPO_OFICIAL:
            codigo, venc, texto = "tempo", -cor, "Derrota automática por tempo."
            break
        if m not in t.legais():
            codigo, venc, texto = "ilegal", -cor, "Derrota automática por lance ilegal."
            break
        san = t.para_san(m)
        if cor == BRANCO:
            san = "%d. %s" % (t.num, san)
        lances_san.append(san)
        tempos[cor].append(gasto)
        profs[cor].append(info["profundidade"])
        t.faz(m)
    if mostrar:
        print(" ".join(lances_san))
        print("\n" + texto)
        for cor, nome in ((BRANCO, "Brancas"), (PRETO, "Pretas")):
            if tempos[cor]:
                print("%s: tempo médio %.2fs, maior %.2fs, profundidade média %.1f" % (
                    nome, sum(tempos[cor]) / len(tempos[cor]), max(tempos[cor]),
                    sum(profs[cor]) / max(1, len(profs[cor]))))
    return {"resultado": codigo, "vencedor": venc, "texto": texto, "lances": lances_san,
            "tempo_max": {c: max(tempos[c]) if tempos[c] else 0 for c in tempos}}


def torneio(motores, mostrar=True):
    """Todos contra todos, dois jogos por confronto (um com cada cor). Vitória 1, empate 0,5."""
    pontos = {m.nome: 0.0 for m in motores}
    for i, a in enumerate(motores):
        for b_ in motores[i + 1:]:
            for brancas, pretas in ((a, b_), (b_, a)):
                res = partida_automatica(brancas, pretas, mostrar=False)
                if res["vencedor"] == BRANCO:
                    pontos[brancas.nome] += 1
                elif res["vencedor"] == PRETO:
                    pontos[pretas.nome] += 1
                else:
                    pontos[brancas.nome] += 0.5
                    pontos[pretas.nome] += 0.5
                if mostrar:
                    print("%s (B) x %s (P): %s" % (brancas.nome, pretas.nome, res["texto"]))
    return pontos

Uma partida de demonstração com 1 segundo por jogada (só para ver o sistema rodando de ponta a ponta). Pode aumentar o tempo se quiser ver jogo mais forte.

In [47]:
res_demo = partida_automatica(Motor(1, nome="IA-A"), Motor(1, nome="IA-B"), max_lances=120)

1. e4 c5 2. Nf3 Nc6 3. d4 cxd4 4. Nxd4 Nf6 5. Nc3 e5 6. Nxc6 dxc6 7. Qxd8+ Kxd8 8. Bg5 Bb4 9. Bxf6+ gxf6 10. Kd2 Be6 11. Be2 Ke7 12. a3 Bc5 13. Ke1 Rhg8 14. Kf1 Rad8 15. Kg1 Rd2 16. Rc1 Bd4 17. Kf1 b5 18. h3 Rd8 19. h4 f5 20. Nb1 Bxb2 21. Nxd2 Bxc1 22. exf5 Bxf5 23. Nf3 Kf6 24. Ne1 Bxa3 25. Bf3 e4 26. g4 exf3 27. gxf5 Rd1 28. Rh3 Bb4 29. Rxf3 Rxe1+ 30. Kg2 Rc1 31. Rb3 a5 32. Rb2 Kxf5 33. h5 Bc3 34. Ra2 Kf4 35. h6 a4 36. Kh3 Kf3 37. Ra1 Bxa1 38. Kh4 Bf6+ 39. Kh3 Rh1#

Xeque-mate! Vencem as pretas.
Brancas: tempo médio 0.60s, maior 1.06s, profundidade média 4.5
Pretas: tempo médio 0.50s, maior 1.07s, profundidade média 4.5


Finais teóricos: a IA joga contra ela mesma a partir de dama+rei x rei e torre+rei x rei. Se a heurística de final estiver boa, sai o xeque-mate em poucos lances.

In [48]:
def teste_finais():
    for fen in ("8/8/8/4k3/8/8/3K4/6Q1 w - - 0 1", "8/8/8/4k3/8/8/3K4/6R1 w - - 0 1"):
        res = partida_automatica(Motor(0.5), Motor(0.5), fen=fen, max_lances=60, mostrar=False)
        print("%-34s -> %s (%d lances)" % (fen, res["texto"], len(res["lances"])))
        assert res["resultado"] == "xeque-mate"


teste_finais()

8/8/8/4k3/8/8/3K4/6Q1 w - - 0 1    -> Xeque-mate! Vencem as brancas. (21 lances)
8/8/8/4k3/8/8/3K4/6R1 w - - 0 1    -> Xeque-mate! Vencem as brancas. (35 lances)


## 8. Interface para jogar contra a IA

Como jogar:
- Digite o lance em **coordenadas** (`e2e4`, `g1f3`, `e7e8q` para promover) ou em **notação padrão** (`e4`, `Nf3`, `exd5`, `O-O`, `e8=Q`).
- Promoção: `q` ou `d` = dama, `r` ou `t` = torre, `b` = bispo, `n` ou `c` = cavalo. Se esquecer, o jogo pergunta.
- Comandos: `ajuda`, `lances` (lista os lances legais), `dica` (a IA sugere um lance), `tabuleiro`, `fen`, `desistir`, `sair`.
- No tabuleiro, letras maiúsculas são as peças brancas e minúsculas as pretas.

In [49]:
NOMES_PT = {1: "peão", 2: "cavalo", 3: "bispo", 4: "torre", 5: "dama", 6: "rei"}


def desenhar(t, virado=False, ultimo=None):
    """Desenha o tabuleiro como imagem usando o python-chess (só para exibição)."""
    b = chess.Board(t.para_fen())
    lance = chess.Move.from_uci(uci(ultimo)) if ultimo else None
    rei_xeque = b.king(b.turn) if b.is_check() else None
    try:
        display(SVG(chess.svg.board(
            b, orientation=chess.BLACK if virado else chess.WHITE,
            lastmove=lance, check=rei_xeque, size=360)))
    except Exception:
        print(b.unicode(borders=True, orientation=chess.BLACK if virado else chess.WHITE))


def motivo_ilegal(t, o, d):
    """Explica por que o lance origem->destino não pode ser feito."""
    p = t.b[o]
    if p == 0:
        return "não há nenhuma peça em %s." % nome_casa(o)
    if p * t.vez < 0:
        return "a peça em %s é do adversário." % nome_casa(o)
    pseudo = [m for m in t.gerar() if m & 255 == o and (m >> 8) & 255 == d]
    if pseudo:
        if t.em_xeque():
            return "seu rei está em xeque e esse lance não resolve."
        return "esse lance deixaria seu rei em xeque (ou o rei passaria por uma casa atacada)."
    if t.b[d] != 0 and t.b[d] * t.vez > 0:
        return "a casa %s já tem uma peça sua." % nome_casa(d)
    return "o %s não anda de %s para %s." % (NOMES_PT[abs(p)], nome_casa(o), nome_casa(d))


def interpretar(t, texto, perguntar=input):
    """Transforma o que o usuário digitou em um lance legal. Retorna (lance, mensagem_de_erro)."""
    legais = t.legais()
    s = texto.strip().replace(" ", "").replace("-", "").replace("–", "")
    if not s:
        return None, "digite um lance (ou 'ajuda')."
    if s.upper().replace("0", "O") in ("OO", "OOO"):
        curto = s.upper().replace("0", "O") == "OO"
        for m in legais:
            if (m >> 20) == F_ROQUE and (((m >> 8) & 255) > (m & 255)) == curto:
                return m, None
        return None, "o roque %s não é possível agora." % ("curto" if curto else "longo")
    mapa_promo = {"q": 5, "d": 5, "r": 4, "t": 4, "b": 3, "n": 2, "c": 2}
    mt = re.fullmatch(r"([a-h][1-8])([a-h][1-8])([qdrtbnc]?)", s.lower())
    if mt:
        o, d = casa_de_nome(mt.group(1)), casa_de_nome(mt.group(2))
        cands = [m for m in legais if m & 255 == o and (m >> 8) & 255 == d]
        if not cands:
            return None, "lance ilegal: " + motivo_ilegal(t, o, d)
        if len(cands) == 1:
            return cands[0], None
        promo = mapa_promo.get(mt.group(3), 0)
        while not promo:
            r = perguntar("Promover para qual peça? (d = dama, t = torre, b = bispo, c = cavalo): ").strip().lower()
            promo = mapa_promo.get(r[:1], 0)
        for m in cands:
            if (m >> 16) & 15 == promo:
                return m, None
    # notação algébrica (SAN)
    def norm(x):
        return x.replace("x", "").replace("+", "").replace("#", "").replace("!", "").replace("?", "").replace("=", "")
    alvo = norm(s)
    for m in legais:
        san = norm(t.para_san(m))
        if alvo == san or (alvo[:1] in "nrqk" and alvo[:1].upper() + alvo[1:] == san):
            return m, None
    return None, "não entendi esse lance. Exemplos: e2e4, g1f3, Nf3, exd5, O-O, e7e8q."


def mostrar_ajuda():
    print("""
Como jogar:
  - Lances em coordenadas: e2e4, g1f3, e7e8q (promoção)  |  ou notação padrão: e4, Nf3, exd5, O-O, e8=Q
  - Comandos: lances | dica | tabuleiro | fen | desistir | sair | ajuda""")


def texto_valor(v):
    """Avaliação em texto, do ponto de vista das brancas (+ brancas, - pretas)."""
    if abs(v) > MATE_LIM:
        n = (MATE - abs(v) + 1) // 2
        return "mate em %d para as %s" % (n, "brancas" if v > 0 else "pretas")
    num = ("%+.2f" % (v / 100.0)).replace(".", ",")
    if abs(v) < 25:
        return "%s (posição equilibrada)" % num
    return "%s (vantagem das %s)" % (num, "brancas" if v > 0 else "pretas")


def jogar(humano_brancas=None, tempo=None, fen=None):
    print("=" * 60)
    print("  XADREZ contra a IA (Minimax + Alfa-Beta)")
    print("=" * 60)
    if humano_brancas is None:
        while True:
            r = input("Quem joga com as BRANCAS (e começa)?  [1] Eu  [2] A IA  [3] Sorteio : ").strip()
            if r in ("1", "2", "3"):
                break
            print("Digite 1, 2 ou 3.")
        humano_brancas = {"1": True, "2": False, "3": random.choice([True, False])}[r]
    if tempo is None:
        r = input("Tempo máximo da IA por jogada em segundos [padrão 10, máximo 110]: ").strip()
        try:
            tempo = float(r.replace(",", ".")) if r else 10.0
        except ValueError:
            print("Valor inválido, usando 10 segundos.")
            tempo = 10.0
    tempo = max(0.5, min(tempo, 110.0))
    t = Tabuleiro(fen)
    ia = Motor(tempo)
    cor_humano = BRANCO if humano_brancas else PRETO
    print("\nVocê joga de %s. A IA joga de %s e pensa no máximo %.0fs por lance." % (
        "BRANCAS" if humano_brancas else "PRETAS", "pretas" if humano_brancas else "brancas", tempo))
    mostrar_ajuda()
    ultimo = None
    while True:
        print()
        desenhar(t, virado=not humano_brancas, ultimo=ultimo)
        r = t.resultado()
        if r:
            print("\n*** FIM DE JOGO: %s ***" % r[2])
            break
        if t.em_xeque():
            print("Xeque!")
        if t.vez == cor_humano:
            entrada = input("\nSeu lance (%s) > " % ("brancas" if t.vez == BRANCO else "pretas"))
            cmd = entrada.strip().lower()
            if cmd in ("sair", "exit", "quit"):
                print("Jogo encerrado.")
                break
            if cmd in ("desistir", "resign"):
                print("Você desistiu. Vitória da IA.")
                break
            if cmd == "ajuda":
                mostrar_ajuda()
                continue
            if cmd == "tabuleiro":
                continue
            if cmd == "fen":
                print(t.para_fen())
                continue
            if cmd == "lances":
                print(", ".join(sorted(t.para_san(m) for m in t.legais())))
                continue
            if cmd == "dica":
                m, info = Motor(min(tempo, 5)).escolher(t)
                print("Dica: %s" % t.para_san(m))
                continue
            m, erro = interpretar(t, entrada)
            if m is None:
                print("Movimento inválido: " + erro)
                continue
            print("Lance válido: %s" % t.para_san(m))
            t.faz(m)
            ultimo = m
        else:
            print("\nA IA está pensando...")
            m, info = ia.escolher(t)
            print("IA jogou: %s  (%s)" % (t.para_san(m), uci(m)))
            if info["livro"]:
                print("   (lance de abertura do livro)")
            else:
                print("   profundidade %d | %.1fs | %d nós | avaliação %s" % (
                    info["profundidade"], info["tempo"], info["nos"], texto_valor(info["valor"])))
            t.faz(m)
            ultimo = m
    return t

## 9. Jogar!

Rode a célula abaixo e responda às perguntas na caixinha que aparece. Para jogar contra a IA de brancas já com 15 segundos por lance, por exemplo: `jogar(humano_brancas=False, tempo=15)`.

**Para a competição:** o jeito mais direto é chamar `jogada_ia(fen, tempo)`, que devolve o lance em texto (ex: `'e2e4'`). Também dá para usar `Motor(tempo).escolher(Tabuleiro(fen))`.

In [ ]:
jogar()

## 10. Decisões de projeto e o que cada requisito do enunciado cobre

- **Regras oficiais:** validamos o gerador com perft em 6 posições de teste famosas (cobrem roque, en passant, promoção, xeque descoberto e peça cravada) e com testes unitários de fim de jogo (mate, afogamento, 50 lances, repetição, material insuficiente).
- **Lances ilegais:** a interface só aceita lances da lista de lances legais e explica o motivo quando recusa. O motor só devolve lances dessa mesma lista, então não gera lance ilegal.
- **Minimax com Alfa-Beta:** o nó de busca tem um ramo que maximiza (brancas) e outro que minimiza (pretas), com `alfa` e `beta`. As melhorias (tabela de transposição, ordenação, quiescência, null move, LMR) servem para cortar mais e chegar mais fundo.
- **Heurística:** material, posição das peças (centro), mobilidade, segurança do rei, estrutura de peões, par de bispos, torres abertas e regras de final.
- **Tempo:** a busca checa o relógio a cada 2048 nós. Estourou o tempo da jogada, volta para o melhor lance da última profundidade completa. O limite máximo configurável é 110 s, abaixo dos 120 s do enunciado. A profundidade 1 sempre termina, então sempre existe uma resposta. Jogadas com um único lance legal e jogadas do livro saem imediatamente.
- **Estabilidade:** o árbitro `partida_automatica` captura exceções e confere legalidade e tempo de cada lance, e foi usado para rodar várias partidas completas durante o desenvolvimento.
- **Limitações:** é Python puro, então a velocidade (nós por segundo) é bem menor que a de motores em C. Por isso a profundidade típica é 4 a 6 lances em 10 s; em finais com poucas peças vai mais fundo. Não há tabela de finais (tablebase).